In [0]:
%pip install pybaseball

from pybaseball import statcast, playerid_lookup
from pyspark.sql import functions as F

#ステップ１:シーズン全体を取得（開幕〜直近）
df = statcast(start_dt = "2026-03-26", end_dt="2026-09-30")
sdf_bronze_ohtani = spark.createDataFrame(df.astype(str))

spark.sql("CREATE SCHEMA IF NOT EXISTS mlb.bronze")
(sdf_bronze_ohtani.write.format("delta").mode("overwrite")
    .saveAsTable("mlb.bronze.statcast_season_raw"))


In [0]:
#ステップ２：大谷選手のIDを取得
lookup = playerid_lookup("ohtani","shohei")
print(lookup[["key_mlbam","name_first","name_last"]])

In [0]:
OHTANI_ID = 660271 #上で取得したIDに置き換え

#既存のSilverクレンジング処理をこのBronzeテーブルにも適用した上で
sdf_silver_ohtani = spark.table("mlb.silver.statcast_clean")

#ステップ２：大谷選手の打席データに絞り込み
sdf_ohtani_batting = sdf_silver_ohtani.filter(F.col("batter") == OHTANI_ID)

In [0]:
from pyspark.sql.functions import weekofyear,year
from pyspark.sql import functions as F

#ステップ３：週単位で集計
sdf_ohtani_weekly = (sdf_ohtani_batting 
    .filter(F.col("launch_speed").isNotNull())
    .withColumn("week",F.date_trunc("week",F.col("game_date")))
    .groupBy("week")
    .agg(
        F.count("*").alias("batted_ball_count"),
        F.avg("launch_speed").alias("avg_exit_velo"),
        F.avg("launch_angle").alias("avg_launch_angle"),
        F.max("launch_speed").alias("max_exit_velo")
    )
    .orderBy("week")
)

#ステップ４：不調前後のフラグ付け
sdf_ohtani_weekly = sdf_ohtani_weekly.withColumn("period",F.when(F.col("week") < "2026-08-18","不調前").otherwise("不調期間"))

(sdf_ohtani_weekly.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("mlb.silver.ohtani_weekly_trend"))

In [0]:
#不調前後の平均値の比較
display(spark.sql("""
    SELECT period,
            AVG(avg_exit_velo) as avg_exit_velo,
            AVG(avg_launch_angle) as avg_launch_angle,
            SUM(batted_ball_count) as total_batted_balls
    FROM mlb.silver.ohtani_weekly_trend
    GROUP BY period
"""))

In [0]:
sdf_ohtani_batting = sdf_silver_ohtani.filter(F.col("batter") == OHTANI_ID)

sdf_ohtani_weekly_detail = (sdf_ohtani_batting
    .filter(F.col("launch_speed").isNotNull())
    .withColumn("week",F.date_trunc("week",F.col("game_date")))
    .withColumn("is_hard_hit",(F.col("launch_speed") >= 95).cast("int"))
    #launch_speed_angle列がある場合（Statcastの公式バレル判定：6=Barrel)
    .withColumn("is_barrel",(F.col("launch_speed_angle") == "6").cast("int"))
    .groupBy("week")
    .agg(
        F.count("*").alias("batted_ball_count"),
        F.avg("launch_speed").alias("avg_exit_velo"),
        F.avg("launch_angle").alias("avg_launch_angle"),
        F.sum("is_hard_hit").alias("hard_hit_count"),
        F.sum("is_barrel").alias("barrel_count")    )
    .withColumn("hard_hit_rate",F.round(F.col("hard_hit_count")/F.col("batted_ball_count") * 100,1))
    .withColumn("barrel_rate", F.round(F.col("barrel_count")/F.col("batted_ball_count") * 100,1))
    .orderBy("week")
)

(sdf_ohtani_weekly_detail.write.format("delta").mode("overwrite").saveAsTable("mlb.gold.ohtani_weekly_detail"))

display(sdf_ohtani_weekly_detail)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.